In [37]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from tabulate import tabulate
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

path = "archive/games.json"

In [32]:
class FileData:
    def __init__(self, path: str):
        self.df = self._load_data(path)

    # -------------------------
    # Loading
    # -------------------------
    def _load_data(self, path: str) -> pd.DataFrame:
        return pd.read_json(path).T

    # -------------------------
    # Feature Engineering
    # -------------------------
    def add_features(self):
        self.df["total_reviews"] = self.df["positive"] + self.df["negative"]
        self.df["wilson_score"] = self._wilson_score(
            self.df["positive"], self.df["negative"]
        )
        return self

    def _wilson_score(self, pos, neg, z=1.96):
        pos = pos.astype(float)
        neg = neg.astype(float)

        n = pos + neg
        n_safe = n.replace(0, np.nan)

        p = pos / n_safe

        score = (
            p + z**2 / (2 * n_safe)
            - z * np.sqrt((p * (1 - p) + z**2 / (4 * n_safe)) / n_safe)
        ) / (1 + z**2 / n_safe)

        return score.fillna(0)

    # -------------------------
    # Filtering
    # -------------------------
    def filter_data(self):
        banned_tags = ["Gore", "Sexual Content", "Nudity"]

        mask_age = self.df["required_age"] <= 17
        mask_content = ~self.df["genres"].str.contains(
            "|".join(banned_tags), case=False, na=False
        )
        mask_reviews = self.df["total_reviews"] >= 50

        self.df = self.df[mask_age & mask_content & mask_reviews]
        return self

    # -------------------------
    # Target Variable
    # -------------------------
    def create_target(self, threshold=0.85):
        self.df["well_received"] = (self.df["wilson_score"] >= threshold).astype(int)
        return self

    def show_class_balance(self):
        print(self.df["well_received"].value_counts(normalize=True))

    # -------------------------
    # Display
    # -------------------------
    def show_top_games(self, n=30):
        cols = ["name", "positive", "negative", "wilson_score"]

        top_df = self.df.sort_values(by="wilson_score", ascending=False)

        print(
            tabulate(
                top_df[cols].head(n),
                headers="keys",
                tablefmt="psql",
                showindex=False,
            )
        )




        
    def linearReg(self):
        df = self.df.copy()

        # -------------------------
        # Basic numeric features
        # -------------------------
        df["windows"] = df["windows"].astype(int)
        df["mac"] = df["mac"].astype(int)
        df["linux"] = df["linux"].astype(int)

        df["num_languages"] = df["supported_languages"].apply(len)

        # -------------------------
        # Genres → one-hot
        # -------------------------
        genre_df = pd.get_dummies(df["genres"].explode()).groupby(level=0).sum()
        genre_df = genre_df.add_prefix("genre_")


        # -------------------------
        # Tags → weighted features
        # -------------------------
        tag_df = pd.json_normalize(df["tags"]).fillna(0)
        tag_df = tag_df.add_prefix("tag_")

        df = df.join(tag_df)
        df = df.join(genre_df)

        # -------------------------
        # Feature matrix
        # -------------------------
        base_features = [
            "price",
            "metacritic_score",
            "dlc_count",
            "recommendations",
            "achievements",
            "required_age",
            "windows",
            "mac",
            "linux",
            "num_languages"
        ]

        X = df[base_features + list(genre_df.columns) + list(tag_df.columns)]
        y = df["well_received"]

        return X, y

In [33]:

print("main")    
data=FileData(path)

data.add_features().filter_data().create_target()
data.show_class_balance()
data.show_top_games()


main
well_received
0    0.712981
1    0.287019
Name: proportion, dtype: float64
+------------------------------------------------+------------+------------+----------------+
| name                                           |   positive |   negative |   wilson_score |
|------------------------------------------------+------------+------------+----------------|
| Kabuto Park                                    |       1035 |          1 |       0.994553 |
| The WereCleaner                                |      10062 |         57 |       0.992709 |
| Aokana - Four Rhythms Across the Blue - EXTRA2 |       1774 |          7 |       0.991909 |
| A Castle Full of Cats                          |       3986 |         23 |       0.991405 |
| Papa's Freezeria Deluxe                        |      11230 |         84 |       0.990818 |
| A Tower Full of Cats                           |       1959 |         10 |       0.990676 |
| A Short Hike                                   |      18904 |        160

In [46]:
X , y = data.linearReg()
X = X.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [47]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)


y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

importance = model.coef_[0]
feature_names = X.columns

for name, coef in sorted(zip(feature_names, importance), key=lambda x: abs(x[1]), reverse=True)[:20]:
    print(name, coef)

Accuracy: 0.7188676157748323
              precision    recall  f1-score   support

           0       0.73      0.97      0.83      4365
           1       0.54      0.10      0.17      1746

    accuracy                           0.72      6111
   macro avg       0.64      0.53      0.50      6111
weighted avg       0.68      0.72      0.64      6111

tag_Short -0.8350175489629387
tag_Action 0.7784907350638673
tag_Puzzle-Platformer -0.7497812890712054
tag_Funny 0.748147034605882
tag_Gore 0.7313247011116593
tag_Early Access -0.6951602848256614
tag_Lore-Rich 0.6804668985949491
tag_Rogue-lite -0.6143762344678817
tag_World War II 0.598870097567786
tag_FPS 0.5910890139099829
tag_Parody  -0.5898959499196182
tag_Stylized 0.5897348956066203
tag_Online Co-Op -0.5741461254800457
tag_Experience 0.5685532921527607
tag_Dark Fantasy -0.5641984668886306
tag_Snow 0.5614898412052323
recommendations 0.561061783794913
tag_Character Action Game 0.5256728788740032
tag_Sports -0.5248861758368072
tag_Grid-